In [0]:
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType


In [0]:
input_filepath ='/Volumes/incremental_sales/default/data/landing/'
checkpoint_path = '/Volumes/incremental_sales/default/metadata/checkpoints/bronze'
schema_path = '/Volumes/incremental_sales/default/metadata/schemas'

In [0]:
schema = StructType ([
    StructField("InvoiceNo", StringType()),
    StructField("StockCode", StringType()),
    StructField("Description", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("InvoiceDate", TimestampType()),
    StructField("UnitPrice", DoubleType()),
    StructField("CustomerID", StringType()),
    StructField("Country", StringType())
])

In [0]:
bronze_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", True)
        .schema(schema)
        .load(input_filepath)
    )

bronze_df = (
    bronze_df
        .withColumn("ingestion_time", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))

    )

(bronze_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("incremental_sales.bronze.sales")
)

In [0]:
%sql
select * from bronze.sales limit 5

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,ingestion_time,source_file
569203,79321,CHILLI LIGHTS,48,2011-10-02T10:32:00.000Z,4.95,16353.0,United Kingdom,2026-08-28T04:51:05.738Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_10.csv
569203,21154,RED RETROSPOT OVEN GLOVE,20,2011-10-02T10:32:00.000Z,1.25,16353.0,United Kingdom,2026-08-28T04:51:05.738Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_10.csv
569204,21790,VINTAGE SNAP CARDS,4,2011-10-02T10:43:00.000Z,0.85,16591.0,United Kingdom,2026-08-28T04:51:05.738Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_10.csv
569204,23284,DOORMAT KEEP CALM AND COME IN,15,2011-10-02T10:43:00.000Z,7.08,16591.0,United Kingdom,2026-08-28T04:51:05.738Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_10.csv
569204,23355,HOT WATER BOTTLE KEEP CALM,4,2011-10-02T10:43:00.000Z,4.95,16591.0,United Kingdom,2026-08-28T04:51:05.738Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_10.csv
